# Autoencoder — Fresh 80-Epoch RTX Training

This notebook performs a **new full training run from scratch** using the validated AE architecture, CASIA splits, 128×128 RGB preprocessing, Adam and MSE loss. It does not run the smoke test and does not overwrite the existing checkpoint.

New best checkpoint: `checkpoints/best_autoencoder_rtx80.pth`. Model selection uses validation MSE; the held-out test split is evaluated only after training.

## 1. Before running

Install an NVIDIA driver and a CUDA-enabled PyTorch build on the RTX laptop. Copy the project folder to the laptop, including `data/splits/` and `data/raw/CASIA2/`. Open this notebook from inside that copied repository.

In [ ]:
%pip install -q numpy pandas Pillow matplotlib scikit-image scikit-learn

## 2. Locate the portable project

In [ ]:
import sys, json, csv, time, zipfile
from pathlib import Path, PureWindowsPath

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src' / 'autoencoder.py').is_file()), None)
assert PROJECT_ROOT is not None, 'Open this notebook from inside the Digital_Evidence repository.'
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Project root:', PROJECT_ROOT)

## 3. Verify RTX CUDA

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA is unavailable. Install CUDA-enabled PyTorch before training.'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('GPU:', GPU_NAME)
print(f'VRAM: {VRAM_GB:.2f} GB')

## 4. Validate CASIA data, fixed splits and mask exclusion

In [ ]:
SPLITS_DIR = PROJECT_ROOT / 'data' / 'splits'
rows_by_split = {}
for name in ('train', 'validation', 'test'):
    with (SPLITS_DIR / f'{name}.csv').open(newline='', encoding='utf-8') as file:
        rows_by_split[name] = list(csv.DictReader(file))
all_rows = sum(rows_by_split.values(), [])
paths = [row['image_path'] for row in all_rows]
counts = {name: len(rows) for name, rows in rows_by_split.items()}
assert counts == {'train': 8830, 'validation': 1892, 'test': 1892}
assert len(paths) == len(set(p.casefold() for p in paths)) == 12614
assert all(not Path(p).is_absolute() and not PureWindowsPath(p).is_absolute() for p in paths)
assert all(Path(p).suffix.lower() != '.png' and 'groundtruth' not in p.lower() for p in paths)
missing = [p for p in paths if not (PROJECT_ROOT / p).is_file()]
assert not missing, f'Missing CASIA images; first five: {missing[:5]}'
print('Dataset validation: PASS', counts)

## 5. Verify preprocessing and unchanged architecture

In [ ]:
from ae_dataset import create_ae_dataloaders
from autoencoder import ConvolutionalAutoencoder

loaders = create_ae_dataloaders(SPLITS_DIR, image_size=128, batch_size=32, num_workers=0, seed=42)
batch = next(iter(loaders['train']))['image']
model = ConvolutionalAutoencoder()
with torch.inference_mode():
    latent = model.encode(batch[:2])
    reconstructed = model(batch[:2])
assert batch.shape == (32, 3, 128, 128)
assert latent.shape == (2, 32, 8, 8)
assert reconstructed.shape == (2, 3, 128, 128)
assert 0 <= float(batch.min()) <= float(batch.max()) <= 1
print('Batch:', tuple(batch.shape), 'range:', (float(batch.min()), float(batch.max())))
print('Latent:', tuple(latent.shape), 'parameters:', sum(p.numel() for p in model.parameters()), 'compression: 24x')

## 6. Fresh 80-epoch configuration

This run starts from random initialization (`initial_checkpoint=None`). The scheduler lowers the learning rate when validation MSE plateaus. Early stopping protects against overfitting, so training may correctly finish before epoch 80. Set `patience=80` only if you specifically need all 80 epochs regardless of validation behavior.

In [ ]:
from argparse import Namespace

TRAIN_ARGS = Namespace(
    splits_dir=SPLITS_DIR,
    initial_checkpoint=None,             # Fresh training, not fine-tuning
    checkpoint_path=PROJECT_ROOT / 'checkpoints' / 'best_autoencoder_rtx80.pth',
    history_path=PROJECT_ROOT / 'results' / 'ae_rtx80_training_history.csv',
    summary_path=PROJECT_ROOT / 'results' / 'ae_rtx80_training_summary.json',
    curve_path=PROJECT_ROOT / 'outputs' / 'ae_rtx80' / 'training_curve.png',
    grid_path=PROJECT_ROOT / 'outputs' / 'ae_rtx80' / 'validation_reconstruction_grid.png',
    image_size=128, batch_size=32, num_workers=0,
    learning_rate=0.001, min_learning_rate=1e-6, weight_decay=1e-6,
    max_epochs=80, patience=12, lr_patience=3, lr_factor=0.5, min_delta=1e-7,
    seed=42, smoke_test=False, smoke_batches=None, require_cuda=True,
)
print(json.dumps(vars(TRAIN_ARGS), indent=2, default=str))

## 7. Train the Autoencoder

This is the real run. It uses CUDA mixed precision, Adam, MSE, validation-based checkpointing, learning-rate scheduling and early stopping.

In [ ]:
import importlib
import train_autoencoder_v2
importlib.reload(train_autoencoder_v2)

training_summary = train_autoencoder_v2.train(TRAIN_ARGS)
print(json.dumps(training_summary, indent=2))

## 8. Training results

In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage
history = pd.read_csv(TRAIN_ARGS.history_path)
display(history)
display(DisplayImage(filename=str(TRAIN_ARGS.curve_path)))
display(DisplayImage(filename=str(TRAIN_ARGS.grid_path)))
best_checkpoint = torch.load(TRAIN_ARGS.checkpoint_path, map_location='cpu', weights_only=True)
print('Best epoch:', best_checkpoint['epoch'])
print('Best validation MSE:', best_checkpoint['validation_loss'])

## 9. Evaluate the best checkpoint on all 1,892 held-out test images

In [ ]:
from evaluate_autoencoder import evaluate

EVAL_ARGS = Namespace(
    splits_dir=SPLITS_DIR, checkpoint_path=TRAIN_ARGS.checkpoint_path,
    per_image_csv=PROJECT_ROOT / 'results' / 'ae_rtx80_test_per_image_metrics.csv',
    metrics_json=PROJECT_ROOT / 'results' / 'ae_rtx80_test_metrics.json',
    reconstruction_grid=PROJECT_ROOT / 'outputs' / 'ae_rtx80' / 'test_reconstruction_grid.png',
    mse_plot=PROJECT_ROOT / 'outputs' / 'ae_rtx80' / 'authentic_vs_tampered_mse.png',
    ssim_plot=PROJECT_ROOT / 'outputs' / 'ae_rtx80' / 'authentic_vs_tampered_ssim.png',
    image_size=128, batch_size=32, num_workers=0, samples_per_class=3, seed=42,
)
test_metrics = evaluate(EVAL_ARGS)
print(json.dumps(test_metrics, indent=2))
display(DisplayImage(filename=str(EVAL_ARGS.reconstruction_grid)))
display(DisplayImage(filename=str(EVAL_ARGS.mse_plot)))

## 10. Compare with the previous validated AE

In [ ]:
baseline_path = PROJECT_ROOT / 'results' / 'ae_test_metrics.json'
if baseline_path.is_file():
    baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
    comparison = pd.DataFrame([
        {'model': 'Previous AE', **{m: baseline['overall'][f'{m}_mean'] for m in ('mse','psnr','ssim')}},
        {'model': 'Fresh RTX-80 AE', **{m: test_metrics['overall'][f'{m}_mean'] for m in ('mse','psnr','ssim')}},
    ])
    display(comparison)
    print('New checkpoint has lower test MSE:', comparison.loc[1, 'mse'] < comparison.loc[0, 'mse'])
else:
    print('Previous metrics unavailable; new complete test metrics are shown above.')
print('Reconstruction statistics are exploratory; this AE is not a forgery classifier.')

## 11. Create the files-to-return ZIP

In [ ]:
RETURN_ZIP = PROJECT_ROOT / 'AE_RTX80_RETURN_ARTIFACTS.zip'
artifact_paths = [
    TRAIN_ARGS.checkpoint_path, TRAIN_ARGS.history_path, TRAIN_ARGS.summary_path,
    TRAIN_ARGS.curve_path, TRAIN_ARGS.grid_path, EVAL_ARGS.per_image_csv,
    EVAL_ARGS.metrics_json, EVAL_ARGS.reconstruction_grid, EVAL_ARGS.mse_plot, EVAL_ARGS.ssim_plot,
]
with zipfile.ZipFile(RETURN_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_paths:
        assert path.is_file(), f'Missing expected artifact: {path}'
        archive.write(path, path.relative_to(PROJECT_ROOT))
print('Copy this ZIP back using the pen drive:', RETURN_ZIP)
print(f'ZIP size: {RETURN_ZIP.stat().st_size / 1024**2:.2f} MB')

## 12. Final summary

In [ ]:
print('GPU:', GPU_NAME)
print('Epochs completed:', training_summary['epochs_completed'])
print('Best epoch:', training_summary['best_epoch'])
print('Best validation MSE:', training_summary['best_validation_loss'])
print('Test MSE:', test_metrics['overall']['mse_mean'])
print('Test PSNR:', test_metrics['overall']['psnr_mean'])
print('Test SSIM:', test_metrics['overall']['ssim_mean'])
print('Training seconds:', training_summary['training_time_seconds'])
print('Checkpoint:', TRAIN_ARGS.checkpoint_path)
print('Return ZIP:', RETURN_ZIP)